# calc_03c. ERA5: observed change (calculations)

Over the years ERA5 and LESFMIP share (1979–2013). Notebooks calc_03a and calc_03b asked whether ERA5 could be one more member of each model's historical ensemble. This notebook asks how far the forced change stands out from natural variability, and whether ERA5's change could have happened without human influence:

1. **Signal-to-noise.** Each model's forced warming against hist-nat's variability from 1850, and the S/N of a straight line through 1979–2013 for every member and for ERA5.
2. **Record rates.** With no change, year *n* is its record's warmest so far with probability 1/*n*. Warming brings more record highs and fewer record lows.
3. **Detection maps.** ERA5's 1979–2013 trend among the hist-nat members' and among the historical members', at every grid point.

Each section has its functions (where it has any), its calculations, then its saves and plots. The results go in `RESULTS_DIR`, beside calc_03a's and calc_03b's; notebook plot_01d opens them. Only notebook calc_01's seasonal means are needed, so this can run before or after calc_03a and calc_03b.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package, plotting_modules and src

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import src.paths as paths
from extant import datatree, era5_evaluation as ev, observed_change as oc, preprocessing as pp
from extant.config import ALPHA, MAIN_EXPERIMENTS, SATELLITE_YEAR_RANGE, VARIABLES
from extant.datatree import skip_empty
from extant.logger import setup_logging
from extant.plots import antarctica, charts, colormap_creators as cc, forced_response as fr, methods
from extant.plots import observed_change as ocp
from extant.statistics import emergence, trend
from plotting_modules.constants import SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Dask Cluster

In [ ]:
#(t): Start a Dask Gateway cluster, or reuse the one already running, and send it the extant package
#(c): Re-run upload_package after editing anything in extant/: the workers run its functions
from extant import jasmin

gateway, cluster, client = jasmin.connect(n_workers=20, wait_for=6)
jasmin.upload_package(client)
client

## Settings

The experiments (`MAIN_EXPERIMENTS`) and the years ERA5 and LESFMIP share (`SATELLITE_YEAR_RANGE`) are in `extant/config.py`, and the order of the seasons (`SEASONS`) in `plotting_modules/constants.py`, the same for every notebook. Only this notebook's own choices are here.

In [ ]:
#(t): The variable: its name and its units once notebook calc_01 has converted it (VARIABLE.name, VARIABLE.units)
VARIABLE = VARIABLES['tas']

In [ ]:
#(t): The grid point of the record rates and of the detection maps' check (the 2.5° cell nearest config.POINT, on the
#(t): Antarctic plateau)
SAMPLE_LAT_LON = dict(lat=-82.5, lon=120.0)

In [ ]:
#(t): The grid point of the S/N quick looks: the nearest to (67°S, 62°W), on the Antarctic Peninsula, where the
#(t): warming is largest
PENINSULA_LAT_LON = dict(lat=-67, lon=-62, method='nearest')

In [ ]:
#(t): The S/N thresholds of the area emerged
SN_THRESHOLDS = [0.5, 1, 2]

## Open the Data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal means, /<model>/<experiment>, on member × year × season × lat × lon), `era5_seasonal_da` (ERA5 on year × season × lat × lon, on the same grid), all in °C, and `RESULTS_DIR`, where the results are saved.

### Sample data

The 3×3 block of grid cells around the point in `tests/data` (two models), put through notebook calc_01's pre-processing.

In [ ]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]

In [ ]:
#(t): Each file of the experiments analysed opened, as a DataTree /<model>/<experiment>
lesfmip_monthly_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                              for path, (model, experiment) in zip(sample_files, sample_names)
                                              if experiment in MAIN_EXPERIMENTS})

In [ ]:
#(t): Seasonal means in °C, through notebook calc_01's pre-processing (pp.seasonal_tree: to °C, cut at 2014-12, complete
#(t): seasons in whole years)
lesfmip_season_tree = pp.seasonal_tree(lesfmip_monthly_tree, VARIABLE.name).compute()

In [ ]:
#(t): ERA5's sample file: monthly tas in K from 1979, on (time, lat, lon)
era5_sample_path = SAMPLE_DIR / 'era5_sample.nc'
era5_monthly_da = xr.open_dataarray(era5_sample_path).load()
era5_sample_path

In [ ]:
#(t): Its seasonal means in °C, on the models' grid (pp.seasonal_era5, as in notebook calc_01)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE.name, like=lesfmip_season_tree)

In [ ]:
#(t): Where the sample's results are saved
RESULTS_DIR = paths.SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

### Full data

The seasonal means notebook calc_01 saved (section 3) in `paths.SCRATCH_DIR`. Opening is lazy.

In [ ]:
#(t): The two stores notebook calc_01 saved
seasonal_path = paths.SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE.name}.zarr'
era5_path = paths.SCRATCH_DIR / 'seasonal' / f'era5_{VARIABLE.name}.zarr'
seasonal_path, era5_path

In [ ]:
#(t): Every model and experiment notebook calc_01 saved
lesfmip_saved_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_saved_tree.filter(lambda node: node.name in MAIN_EXPERIMENTS)

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE.name].load()

In [ ]:
#(t): Where the full data's results are saved
RESULTS_DIR = paths.SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

### What was opened

Whichever section ran. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): 'CanESM5's ensemble mean of each experiment at the point in DJF, on (experiment, year)
example_point_ds = datatree.tree_to_dataset(lesfmip_season_tree['CanESM5'].sel(SAMPLE_LAT_LON).sel(season='DJF')
                                            .mean('member'), dims=['experiment'])

In [ ]:
#(t): Quick look: those ensemble means, and ERA5 (black)
example_point_ds[VARIABLE.name].plot(hue='experiment')
era5_seasonal_da.sel(SAMPLE_LAT_LON).sel(season='DJF').plot(color='k');

In [ ]:
#(t): Quick look: 'CanESM5's historical members at the point, each season
lesfmip_season_tree['CanESM5/historical'][VARIABLE.name].sel(SAMPLE_LAT_LON).sel(season=SEASONS).plot(
    hue='member', col='season', col_wrap=2, add_legend=False);

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=MAIN_EXPERIMENTS)

In [ ]:
#(t): The members: each model (rows) and experiment (columns), grey where there are none
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.9 * member_counts_df.shape[1] + 0.5,
                                                  panel_height=0.4 * member_counts_df.shape[0] + 0.3, left_margin=1.8),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
counts_image = methods.plot_member_counts(member_counts_df, ax=axes[0, 0])
fig.colorbar(counts_image, ax=axes, location='bottom', shrink=0.6, label='Members')
fig.suptitle('LESFMIP members per model and experiment', x=0.01, ha='left');

## Matching ERA5 and LESFMIP Time Periods

LESFMIP and ERA5 over the years they share, `SATELLITE_YEAR_RANGE` (ERA5 runs on past 2013).

In [ ]:
#(t): LESFMIP over the years shared with ERA5
lesfmip_obs_tree = lesfmip_season_tree.sel(year=SATELLITE_YEAR_RANGE)

In [ ]:
#(t): ERA5 over the same years, on (year, season, lat, lon)
era5_obs_da = era5_seasonal_da.sel(year=SATELLITE_YEAR_RANGE)

## 1. Signal-to-Noise Ratio

Two S/N, both from `extant/statistics/emergence.py`:

- **Forced S/N, 1850–2013** (`emergence.proper_sn`), for each model. Signal: the historical ensemble mean's anomaly from hist-nat's climatology, smoothed with LOWESS (81-year window). Noise: hist-nat's standard deviation over every year and member. All of a model's members go into the signal, so it is the forced change alone; ERA5 has no ensemble, so it has no forced S/N.
- **Linear S/N, 1979–2013** (`emergence.linear_sn`), for each member and ERA5 alike. Signal: the rise of the series' straight line since 1979. Noise: the standard deviation of the series about the line.

Then the percent of the area where the S/N is greater than `SN_THRESHOLDS` (`emergence.emergence_surface_area_percent`), the spread of the S/N over the area, and how much of each model's forced warming came before 1979. These are quick to work out from the saved S/N, so they are not saved.

### Calculations

**Forced S/N**

In [ ]:
#(t): Each model's historical members, 1850–2013
historical_tree = lesfmip_season_tree.match('*/historical')

In [ ]:
#(t): Each model's hist-nat members, in its historical's place in the tree, so the two trees line up node for node
histnat_like_tree = datatree.hist_nat_like(lesfmip_season_tree).match('*/historical')

In [ ]:
#(t): Example: 'CanESM5's forced S/N at every grid point, on (season, year, lat, lon)
example_proper_sn_ds = emergence.proper_sn(lesfmip_season_tree['CanESM5/historical'][VARIABLE.name],
                                           histnat_da=lesfmip_season_tree['CanESM5/hist-nat'][VARIABLE.name]).compute()

In [ ]:
#(t): ... and the anomaly its signal smooths: the ensemble mean minus hist-nat's climatology (over years and members)
example_anomaly_da = (lesfmip_season_tree['CanESM5/historical'][VARIABLE.name].mean('member')
                      - lesfmip_season_tree['CanESM5/hist-nat'][VARIABLE.name].mean(['year', 'member']))

In [ ]:
#(t): The forced S/N step by step, 'CanESM5' at the Peninsula point in DJF: a) the anomaly (grey) and its smoothed
#(t): signal (black), with hist-nat's noise either side of 0 (dashed); b) the S/N, signal / noise
example_point_anomaly_da = example_anomaly_da.sel(**PENINSULA_LAT_LON).sel(season='DJF')
example_point_sn_ds = example_proper_sn_ds.sel(**PENINSULA_LAT_LON).sel(season='DJF')

fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=2, n_legends=0), layout='constrained')
axes = fig.add_gridspec(1, 2).subplots(squeeze=False)[0]
axes[0].plot(example_point_anomaly_da['year'], example_point_anomaly_da, color='0.6')
axes[0].plot(example_point_sn_ds['year'], example_point_sn_ds['signal'], color='k')
for sign in (-1, 1):
    axes[0].axhline(sign * float(example_point_sn_ds['noise']), color='k', ls='--', lw=0.8)
axes[0].set(title='a) Anomaly, signal and noise', ylabel=VARIABLE.units)
axes[1].plot(example_point_sn_ds['year'], example_point_sn_ds['sn'], color='k')
axes[1].set(title='b) S/N');

In [ ]:
%%time
#(t): Every model's forced S/N, historical against its own hist-nat
proper_sn_tree = xr.map_over_datasets(
    lambda historical_ds, histnat_ds: emergence.proper_sn(historical_ds, histnat_da=histnat_ds),
    historical_tree, histnat_like_tree)

In [ ]:
#(t): /<model>/historical becomes /<model> (every path ends in historical)
proper_sn_tree = datatree.streamline_tree(proper_sn_tree)

In [ ]:
#(t): On (model, season, year, lat, lon): no members left, so the models stack into one Dataset
proper_sn_ds = datatree.tree_to_dataset(proper_sn_tree, dims=['model']).compute()
proper_sn_ds

In [ ]:
#(t): Check: the example is 'CanESM5's (largest difference of the S/N: 0)
float(abs(proper_sn_ds['sn'].sel(model='CanESM5') - example_proper_sn_ds['sn']).max())

**Linear S/N, 1979–2013**

In [ ]:
#(t): ERA5's at every grid point, on (season, year, lat, lon): its line (fit, gradient), the line's rise since 1979
#(t): (linear_signal), the series about the line (linear_detrend), its standard deviation (linear_noise), and the S/N
era5_linear_sn_ds = emergence.linear_sn(era5_obs_da)

In [ ]:
#(t): The linear S/N step by step, ERA5 at the Peninsula point in DJF: a) the series (grey) and its line (black);
#(t): b) the series about the line, with its standard deviation either side of 0 (dashed); c) the S/N, the line's rise
#(t): over that standard deviation
era5_point_sn_ds = era5_linear_sn_ds.sel(**PENINSULA_LAT_LON).sel(season='DJF')
era5_point_series_da = era5_obs_da.sel(**PENINSULA_LAT_LON).sel(season='DJF')

fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=3, n_legends=0), layout='constrained')
axes = fig.add_gridspec(1, 3).subplots(squeeze=False)[0]
axes[0].plot(era5_point_series_da['year'], era5_point_series_da, color='0.6')
axes[0].plot(era5_point_sn_ds['year'], era5_point_sn_ds['fit'], color='k')
axes[0].set(title='a) ERA5 and its line', ylabel=VARIABLE.units)
axes[1].plot(era5_point_sn_ds['year'], era5_point_sn_ds['linear_detrend'], color='0.6')
for sign in (-1, 1):
    axes[1].axhline(sign * float(era5_point_sn_ds['linear_noise']), color='k', ls='--', lw=0.8)
axes[1].set(title='b) About the line, and the noise', ylabel=VARIABLE.units)
axes[2].plot(era5_point_sn_ds['year'], era5_point_sn_ds['linear_sn'], color='k')
axes[2].set(title='c) S/N');

In [ ]:
#(t): Each model's historical members over the shared years, /<model>/historical becoming /<model>
historical_obs_tree = datatree.streamline_tree(lesfmip_obs_tree.match('*/historical'))

In [ ]:
#(t): Every member's linear S/N, the same way as ERA5's
linear_sn_tree = historical_obs_tree.map_over_datasets(skip_empty(emergence.linear_sn))

In [ ]:
#(t): Each model's median member S/N, on (model, season, year, lat, lon)
median_linear_sn_ds = datatree.reduce_to_dataset(linear_sn_tree, lambda ds: ds[['linear_sn']].median('member'),
                                                 dims=['model']).compute()
median_linear_sn_ds

**Area emerged, spread over the area, and the warming before 1979** (not saved)

In [ ]:
#(t): Percent of the area where each model's forced S/N is greater than each threshold, on (threshold, model, season, year)
proper_sn_area_da = emergence.emergence_surface_area_percent(proper_sn_ds['sn'], SN_THRESHOLDS)

In [ ]:
#(t): ... the models' median member's linear S/N
median_linear_sn_area_da = emergence.emergence_surface_area_percent(median_linear_sn_ds['linear_sn'], SN_THRESHOLDS)

In [ ]:
#(t): ... and ERA5's
era5_linear_sn_area_da = emergence.emergence_surface_area_percent(era5_linear_sn_ds['linear_sn'], SN_THRESHOLDS)

In [ ]:
#(t): Each grid cell's share of the area: cos(latitude)
area_weights_da = np.cos(np.deg2rad(proper_sn_ds['lat']))

In [ ]:
#(t): The forced signal's and S/N's quantiles over the area, on (quantile, model, season, year)
proper_sn_area_quantiles_ds = proper_sn_ds[['signal', 'sn']].weighted(area_weights_da).quantile(
    [0.05, 0.25, 0.5, 0.75, 0.95], dim=['lat', 'lon'])

In [ ]:
#(t): Each model's forced warming by 1979, and by 2013
signal_1979_da = proper_sn_ds['signal'].sel(year=SATELLITE_YEAR_RANGE.start)
signal_2013_da = proper_sn_ds['signal'].sel(year=SATELLITE_YEAR_RANGE.stop)

In [ ]:
#(t): ... the warming over 1979–2013
signal_since_1979_da = signal_2013_da - signal_1979_da

In [ ]:
#(t): ... as a percent of the warming by 2013
signal_since_1979_percent_da = 100 * signal_since_1979_da / signal_2013_da

### Save

In [ ]:
#(t): Every model's forced S/N: signal, noise and S/N
proper_sn_path = RESULTS_DIR / 'proper_sn.zarr'
proper_sn_ds.to_zarr(proper_sn_path, mode='w')
proper_sn_path

In [ ]:
#(t): ERA5's linear S/N: the line's rise, the noise and the S/N
era5_linear_sn_path = RESULTS_DIR / 'era5_linear_sn.zarr'
era5_linear_sn_ds[['linear_signal', 'linear_noise', 'linear_sn']].to_zarr(era5_linear_sn_path, mode='w')
era5_linear_sn_path

In [ ]:
#(t): Each model's median member's linear S/N
median_linear_sn_path = RESULTS_DIR / 'median_linear_sn.zarr'
median_linear_sn_ds.to_zarr(median_linear_sn_path, mode='w')
median_linear_sn_path

### Plots

In [ ]:
#(t): The colour scales shared with the other notebooks (fr.colour_scales): here the S/N's
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): Quick look: every model's forced S/N at the Peninsula point, each season
proper_sn_ds['sn'].sel(**PENINSULA_LAT_LON).sel(season=SEASONS).plot(hue='model', col='season');

In [ ]:
#(t): The forced S/N over the area: its median (black), 25–75% and 5–95% (bands), each model (rows) and season (columns)
area_sn_quantiles_da = proper_sn_area_quantiles_ds['sn'].sel(season=SEASONS)
n_rows, n_cols = area_sn_quantiles_da.sizes['model'], area_sn_quantiles_da.sizes['season']
fig = plt.figure(figsize=charts.chart_figure_size(n_rows, n_cols, n_legends=0, panel_height=0.7), layout='constrained')
axes = fig.add_gridspec(n_rows, n_cols).subplots(sharex=True, sharey=True, squeeze=False)

charts.quantile_bands_grid(area_sn_quantiles_da, [(0.05, 0.95), (0.25, 0.75)], 'year', row_dim='model',
                           col_dim='season', axes=axes)
charts.lines_grid(area_sn_quantiles_da.sel(quantile=0.5), 'year', row_dim='model', col_dim='season', axes=axes,
                  color='k')

for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season)
for ax, model in zip(axes[:, 0], area_sn_quantiles_da['model'].values):
    ax.set_ylabel(model)
fig.suptitle('Forced S/N over the area', x=0.01, ha='left');

In [ ]:
#(t): The same for the forced signal
area_signal_quantiles_da = proper_sn_area_quantiles_ds['signal'].sel(season=SEASONS)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows, n_cols, n_legends=0, panel_height=0.7), layout='constrained')
axes = fig.add_gridspec(n_rows, n_cols).subplots(sharex=True, sharey=True, squeeze=False)

charts.quantile_bands_grid(area_signal_quantiles_da, [(0.05, 0.95), (0.25, 0.75)], 'year', row_dim='model',
                           col_dim='season', axes=axes)
charts.lines_grid(area_signal_quantiles_da.sel(quantile=0.5), 'year', row_dim='model', col_dim='season', axes=axes,
                  color='k')

for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season)
for ax, model in zip(axes[:, 0], area_signal_quantiles_da['model'].values):
    ax.set_ylabel(model)
fig.suptitle(f'Forced signal over the area ({VARIABLE.units})', x=0.01, ha='left');

In [ ]:
#(t): Quick look: percent of the area with a forced S/N greater than 1, every model
proper_sn_area_da.sel(threshold=1).sel(season=SEASONS).plot(hue='model', col='season', col_wrap=2);

In [ ]:
#(t): Quick look: percent of the area with a linear S/N greater than 1, each model's median member and ERA5 (black)
area_grid = median_linear_sn_area_da.sel(threshold=1).sel(season=SEASONS).plot(hue='model', col='season', col_wrap=2)
for ax, season in zip(area_grid.axs.flat, SEASONS):
    era5_linear_sn_area_da.sel(threshold=1, season=season).plot(ax=ax, color='k', lw=3)

In [ ]:
#(t): The forced warming's colours: blues below 0, oranges from 0 to 1, reds from 1 to 2 (four steps each)
signal_levels, signal_cmap = cc.banded_colormap([
    cc.Band(-0.5, 0, 'Blues_r', 0.5, 1, n_steps=4),
    cc.Band(0, 1, 'Oranges', 0.0, 0.3, n_steps=4),
    cc.Band(1, 2, 'Reds', 0.4, 1.0, n_steps=4),
], extend='both')

In [ ]:
#(t): Each model's forced warming by 1979 (upper row) and over 1979–2013 (lower row)
models = proper_sn_ds['model'].values
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(models), n_cols=4, n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, 2 * len(models), 4)
cax = antarctica.colorbar_axis_below(axes[-1, :])

antarctica.heatmaps_of_antarctica(signal_1979_da.sel(season=SEASONS), 'model', 'season', axes=axes[::2],
                                  levels=signal_levels, cmap=signal_cmap, cax=cax, label=f'Forced warming ({VARIABLE.units})')
antarctica.heatmaps_of_antarctica(signal_since_1979_da.sel(season=SEASONS), 'model', 'season', axes=axes[1::2],
                                  levels=signal_levels, cmap=signal_cmap)

antarctica.label_columns(axes[0, :], SEASONS)
antarctica.label_rows(axes, [f'{model}\n{period}' for model in models for period in ('by 1979', '1979–2013')]);

In [ ]:
#(t): Each model's warming over 1979–2013, as a percent of its forced warming by 2013
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=4, n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, len(models), 4)
cax = antarctica.colorbar_axis_below(axes[-1, :])

antarctica.heatmaps_of_antarctica(signal_since_1979_percent_da.sel(season=SEASONS), 'model', 'season', axes=axes,
                                  levels=np.arange(-120, 140, 20), cax=cax, label='Warming over 1979–2013 (%)')

antarctica.label_columns(axes[0, :], SEASONS)
antarctica.label_rows(axes, list(models));

In [ ]:
#(t): Each model's forced S/N in 2013
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=4, n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, len(models), 4)
cax = antarctica.colorbar_axis_below(axes[-1, :])

antarctica.heatmaps_of_antarctica(proper_sn_ds['sn'].sel(year=SATELLITE_YEAR_RANGE.stop).sel(season=SEASONS), 'model',
                                  'season', axes=axes, cax=cax, **scales.sn.colours())

antarctica.label_columns(axes[0, :], SEASONS)
antarctica.label_rows(axes, list(models));

In [ ]:
#(t): ERA5's linear S/N in 2013 and its two parts: the line's rise over 1979–2013, the noise, and the S/N
era5_linear_sn_2013_ds = era5_linear_sn_ds.sel(year=SATELLITE_YEAR_RANGE.stop).sel(season=SEASONS)

fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=4, n_colorbar_rows=3), layout='constrained')
axes = antarctica.map_axes(fig, 3, 4)

antarctica.heatmaps_of_antarctica(era5_linear_sn_2013_ds['linear_signal'], None, 'season', axes=axes[0, :],
                                  cax=antarctica.colorbar_axis_below(axes[0, :]), levels=np.arange(-2, 2.2, 0.2),
                                  cmap='RdBu_r', label=f'Signal ({VARIABLE.units})')
antarctica.heatmaps_of_antarctica(era5_linear_sn_2013_ds['linear_noise'], None, 'season', axes=axes[1, :],
                                  cax=antarctica.colorbar_axis_below(axes[1, :]), levels=np.arange(0, 2.2, 0.2),
                                  cmap='YlOrBr', extend='max', label=f'Noise ({VARIABLE.units})')
antarctica.heatmaps_of_antarctica(era5_linear_sn_2013_ds['linear_sn'], None, 'season', axes=axes[2, :],
                                  cax=antarctica.colorbar_axis_below(axes[2, :]), **scales.sn.colours())

antarctica.label_columns(axes[0, :], SEASONS)
antarctica.label_rows(axes, ['Signal', 'Noise', 'S/N']);

In [ ]:
#(t): The linear S/N in 2013: ERA5 (top row) and each model's median member
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models) + 1, n_cols=4, n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, len(models) + 1, 4)
cax = antarctica.colorbar_axis_below(axes[-1, :])

antarctica.heatmaps_of_antarctica(era5_linear_sn_2013_ds['linear_sn'], None, 'season', axes=axes[0, :], cax=cax,
                                  **scales.sn.colours())
antarctica.heatmaps_of_antarctica(median_linear_sn_ds['linear_sn'].sel(year=SATELLITE_YEAR_RANGE.stop)
                                  .sel(season=SEASONS), 'model', 'season', axes=axes[1:, :], **scales.sn.colours())

antarctica.label_columns(axes[0, :], SEASONS)
antarctica.label_rows(axes, ['ERA5'] + list(median_linear_sn_ds['model'].values));

## 2. Record Rates

Cumulative record highs and lows through 1979–2013 at the point, for ERA5 and 'CanESM5's hist-nat and historical members, the four seasons pooled so there are enough records to compare (`oc.record_curves`). One model at one point, so it is drawn here and not saved.

### Calculations

In [ ]:
#(t): 'CanESM5's hist-nat and historical members at the point, on (member, year, season)
example_point_histnat_da = lesfmip_obs_tree['CanESM5/hist-nat'][VARIABLE.name].sel(SAMPLE_LAT_LON)
example_point_historical_da = lesfmip_obs_tree['CanESM5/historical'][VARIABLE.name].sel(SAMPLE_LAT_LON)

In [ ]:
#(t): ERA5 at the point, on (year, season)
era5_point_da = era5_obs_da.sel(SAMPLE_LAT_LON)

In [ ]:
#(t): The cumulative records: ERA5's (obs), each member's (members) and a stationary series' (expected)
point_record_curves_ds = oc.record_curves({'hist-nat': example_point_histnat_da,
                                           'historical': example_point_historical_da}, era5_point_da)
point_record_curves_ds

### Plots

In [ ]:
#(t): Cumulative record highs and lows at the point, all seasons pooled: ERA5 against chance (dashed) and against
#(t): the members of each experiment (bands)
record_kinds = list(point_record_curves_ds['kind'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(record_kinds), n_legends=1, panel_width=4.5),
                 layout='constrained')
axes = fig.add_gridspec(1, len(record_kinds)).subplots(squeeze=False)
for ax, kind in zip(axes[0], record_kinds):
    ocp.plot_record_rates(point_record_curves_ds, kind, ax=ax)
    ax.set_title(f'Record {kind}')
axes[0, 0].set_ylabel('Cumulative number of records')
fig.legend(handles=ocp.record_rate_legend_handles(), loc='outside lower center', ncols=4, frameon=False);

## 3. Detection Maps

ERA5's 1979–2013 trend (`trend.trend`, °C per decade) among each experiment's members' (`ev.locate`) at every grid point, against hist-nat and against historical. Each grid point then falls in one of four classes (`oc.DETECTION_CLASSES`): detected and consistent (outside hist-nat, inside historical), not detected and consistent, detected and inconsistent, not detected and inconsistent.

### Functions

In [ ]:
def map_trend_test(ds, era5_trend_da):
    """Where ERA5's 1979-2013 trend falls among one experiment's members' trends, at every grid point.

    Args:
        ds (xr.Dataset): One model's experiment: tas on (member, year, season, lat, lon), over SATELLITE_YEAR_RANGE.
        era5_trend_da (xr.DataArray): ERA5's trend on (season, lat, lon).

    Returns:
        xr.Dataset: ev.locate's results (without the members) on (season, lat, lon).
    """
    members_trend_da = trend.trend(ds[VARIABLE.name].load(), dim='year')
    return ev.locate(members_trend_da, era5_trend_da).drop_dims('member')

### Calculations

In [ ]:
#(t): ERA5's trend at every grid point, on (season, lat, lon)
era5_trend_da = trend.trend(era5_obs_da, dim='year')

In [ ]:
%%time
#(t): Example: 'CanESM5's historical
example_trend_map_ds = map_trend_test(lesfmip_obs_tree['CanESM5/historical'].to_dataset(), era5_trend_da)
example_trend_map_ds

In [ ]:
#(t): The same test from the point's own series: ERA5's trend among 'CanESM5's historical members' at the point
point_trend_test_ds = ev.locate(trend.trend(example_point_historical_da, dim='year'),
                                trend.trend(era5_point_da, dim='year'))

In [ ]:
#(t): Check: at the point, the map gives the point's test (largest difference of ERA5's percentile: 0)
float(abs(example_trend_map_ds['percentile'].sel(SAMPLE_LAT_LON) - point_trend_test_ds['percentile']).max())

In [ ]:
#(t): Every model's historical and hist-nat
detection_tree = lesfmip_obs_tree.filter(lambda node: node.name in ('historical', 'hist-nat'))

In [ ]:
%%time
#(t): The test for each of them, on (model, experiment, season, lat, lon)
map_trend_tests_ds = datatree.reduce_to_dataset(detection_tree, lambda ds: map_trend_test(ds, era5_trend_da))

In [ ]:
#(t): Each grid point's class, on (model, season, lat, lon)
detection_class_da = oc.detection_class(map_trend_tests_ds.sel(experiment='historical'),
                                        map_trend_tests_ds.sel(experiment='hist-nat'))

In [ ]:
#(t): The classes and ERA5's percentile among each experiment's members, as notebooks plot_01d and plot_06 open them
detections_ds = xr.Dataset({
    'detection_class': detection_class_da,
    'historical_percentile': map_trend_tests_ds['percentile'].sel(experiment='historical', drop=True),
    'hist-nat_percentile': map_trend_tests_ds['percentile'].sel(experiment='hist-nat', drop=True),
})
detections_ds

### Save

In [ ]:
#(t): The classes and percentiles
detections_path = RESULTS_DIR / 'detections.zarr'
detections_ds.to_zarr(detections_path, mode='w')
detections_path

### Plots

In [ ]:
#(t): Quick look: 'CanESM5's class at each grid point, every season
detection_class_da.sel(model='CanESM5').sel(season=SEASONS).plot(x='lon', y='lat', col='season');